# Word2Vec with Gensim

This notebook walks through **Word2Vec** from first principles using Gensim — including training your own model.

## What is Word2Vec?

Word2Vec (Mikolov et al., 2013) learns dense word representations by predicting words from context.  
The core idea: **"You shall know a word by the company it keeps."**

Two training architectures:

| Architecture | Predicts | Best for |
|---|---|---|
| **Skip-gram** (`sg=1`) | Context words from center word | Rare words, small datasets |
| **CBOW** (`sg=0`) | Center word from context window | Frequent words, large datasets |

```
Skip-gram:  [the, ___, fox]  ──▶  quick, brown
CBOW:       [the, quick, brown, ___]  ──▶  fox
```

Training objective: adjust word vectors so that **words appearing in similar contexts** end up close in vector space.

In [ ]:
!pip install -q gensim matplotlib scikit-learn numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from collections import Counter
import time
import logging

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess

plt.style.use('ggplot')
%matplotlib inline

print(f"Gensim version: {gensim.__version__}")

---
## 1. Train on a Small Custom Corpus

We start with a tiny hand-crafted corpus to see exactly what the model learns.  
In Word2Vec, the input is a **list of tokenized sentences** (a list of lists of strings).

In [ ]:
sentences = [
    # Animals
    ["the", "dog", "barked", "at", "the", "cat"],
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "chased", "the", "cat"],
    ["the", "puppy", "played", "with", "the", "kitten"],
    ["dogs", "and", "cats", "are", "common", "pets"],
    ["the", "puppy", "barked", "loudly"],
    ["the", "kitten", "meowed", "softly"],
    # Food
    ["i", "ate", "pizza", "for", "lunch"],
    ["she", "loves", "eating", "pasta", "and", "pizza"],
    ["the", "restaurant", "serves", "delicious", "pizza"],
    ["we", "cooked", "spaghetti", "and", "pasta", "for", "dinner"],
    ["italian", "food", "includes", "pizza", "pasta", "and", "risotto"],
    # Royalty
    ["the", "king", "ruled", "the", "kingdom"],
    ["the", "queen", "sat", "on", "the", "throne"],
    ["the", "king", "and", "queen", "governed", "wisely"],
    ["the", "prince", "became", "king"],
    ["the", "princess", "became", "queen"],
    ["the", "man", "was", "crowned", "king"],
    ["the", "woman", "was", "crowned", "queen"],
    # Tech
    ["machine", "learning", "uses", "neural", "networks"],
    ["deep", "learning", "is", "a", "subset", "of", "machine", "learning"],
    ["neural", "networks", "learn", "from", "data"],
    ["python", "is", "used", "for", "machine", "learning"],
    ["data", "science", "applies", "machine", "learning"],
]

# Check vocabulary size
all_words = [w for s in sentences for w in s]
vocab = Counter(all_words)
print(f"Total tokens: {len(all_words)}")
print(f"Unique words: {len(vocab)}")
print(f"Most common: {vocab.most_common(10)}")

In [ ]:
# Train a Skip-gram Word2Vec model on our small corpus
model_small = Word2Vec(
    sentences=sentences,
    vector_size=50,     # embedding dimension
    window=3,           # context window size (words to the left/right)
    min_count=1,        # include words seen at least this many times
    sg=1,               # 1 = Skip-gram, 0 = CBOW
    negative=5,         # negative sampling: 5 noise words per positive sample
    epochs=200,         # training passes over the corpus
    seed=42,
    workers=4,
)

print(f"Vocabulary size: {len(model_small.wv)}")
print(f"Vector dimension: {model_small.wv.vector_size}")
print(f"Vector for 'king': {model_small.wv['king'][:8]}...")

In [ ]:
# Words similar to 'king'
print("Most similar to 'king':")
for word, score in model_small.wv.most_similar('king', topn=5):
    print(f"  {word:15s} {score:.4f}")

print()
print("Most similar to 'dog':")
for word, score in model_small.wv.most_similar('dog', topn=5):
    print(f"  {word:15s} {score:.4f}")

print()
print("Most similar to 'pizza':")
for word, score in model_small.wv.most_similar('pizza', topn=5):
    print(f"  {word:15s} {score:.4f}")

In [ ]:
# Vector arithmetic: king - man + woman = ?
# (small corpus, so results may be rough)
result = model_small.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("king - man + woman =")
for word, score in result:
    print(f"  {word:15s} {score:.4f}")

print()

# Cosine similarities between pairs
pairs = [
    ('dog', 'cat'),
    ('dog', 'puppy'),
    ('king', 'queen'),
    ('pizza', 'pasta'),
    ('dog', 'pizza'),
]
print("Cosine similarities:")
for w1, w2 in pairs:
    sim = model_small.wv.similarity(w1, w2)
    print(f"  {w1:10s} <-> {w2:10s}: {sim:.4f}")

---
## 2. Train on Real Data: the `text8` Corpus

`text8` is the first 100M characters of Wikipedia — a standard Word2Vec benchmark dataset.

Gensim can download it directly. Training on it takes a few minutes.

In [ ]:
# Download text8 (this may take a minute the first time)
print("Loading text8 corpus...")
corpus = api.load('text8')

# text8 is an iterable of sentences (lists of tokens)
# Load it into memory so we can inspect and reuse it
sentences_text8 = list(corpus)

total_tokens = sum(len(s) for s in sentences_text8)
total_sentences = len(sentences_text8)
vocab_count = len(Counter(w for s in sentences_text8 for w in s))

print(f"Sentences:    {total_sentences:,}")
print(f"Total tokens: {total_tokens:,}")
print(f"Unique words: {vocab_count:,}")
print(f"Sample sentence: {sentences_text8[0][:10]}...")

In [ ]:
# Enable logging to see training progress
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

print("Training Word2Vec on text8...")
start = time.time()

model = Word2Vec(
    sentences=sentences_text8,
    vector_size=100,    # 100-dimensional vectors
    window=5,           # ±5 word context window
    min_count=5,        # ignore words appearing fewer than 5 times
    sg=1,               # Skip-gram
    negative=5,         # 5 negative samples per update
    epochs=5,
    seed=42,
    workers=4,
)

elapsed = time.time() - start
logging.disable(logging.CRITICAL)  # suppress logs after training

print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Vocabulary size: {len(model.wv):,}")
print(f"Vector dimension: {model.wv.vector_size}")

---
## 3. Exploring the Trained Model

In [ ]:
# Nearest neighbors
words_to_check = ['king', 'computer', 'france', 'music', 'apple']

for word in words_to_check:
    if word in model.wv:
        print(f"\nMost similar to '{word}':")
        for w, score in model.wv.most_similar(word, topn=5):
            print(f"  {w:20s} {score:.4f}")

In [ ]:
# Word analogies via vector arithmetic
# king - man + woman = queen

def analogy(model, x1, x2, y1, topn=1):
    """Solves: x1 is to x2 as y1 is to ?"""
    result = model.wv.most_similar(positive=[y1, x2], negative=[x1], topn=topn)
    return result

analogies = [
    ('man',     'king',    'woman'),    # classic royalty
    ('paris',   'france',  'berlin'),   # capitals
    ('paris',   'france',  'tokyo'),    # capitals
    ('good',    'better',  'bad'),      # comparatives
    ('walk',    'walking', 'swim'),     # gerunds
    ('small',   'smaller', 'large'),    # comparatives
    ('doctor',  'man',     'nurse'),    # gender roles (may reveal bias)
]

print(f"{'x1':10s}  {'x2':10s}  {'y1':10s}  =>  Answer")
print("-" * 55)
for x1, x2, y1 in analogies:
    try:
        result = analogy(model, x1, x2, y1)[0]
        print(f"{x1:10s}  {x2:10s}  {y1:10s}  =>  {result[0]} ({result[1]:.3f})")
    except KeyError as e:
        print(f"{x1:10s}  {x2:10s}  {y1:10s}  =>  [word not in vocab: {e}]")

In [ ]:
# Which word doesn't belong?
groups = [
    ['cat', 'dog', 'horse', 'banana'],
    ['paris', 'london', 'berlin', 'pizza'],
    ['football', 'tennis', 'swimming', 'chocolate'],
]

for group in groups:
    # filter to words in vocab
    valid = [w for w in group if w in model.wv]
    if len(valid) >= 3:
        odd = model.wv.doesnt_match(valid)
        print(f"{valid}  =>  odd one out: '{odd}'")

In [ ]:
# Compute pairwise cosine similarities for a set of related words
country_words = ['france', 'germany', 'italy', 'spain', 'japan', 'china']
country_words = [w for w in country_words if w in model.wv]

n = len(country_words)
sim_matrix = np.zeros((n, n))
for i, w1 in enumerate(country_words):
    for j, w2 in enumerate(country_words):
        sim_matrix[i, j] = model.wv.similarity(w1, w2)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0.3, vmax=1.0)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(country_words, rotation=45)
ax.set_yticklabels(country_words)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center', fontsize=9)
ax.set_title("Pairwise Similarity: Country Words")
plt.tight_layout()
plt.show()

---
## 4. Skip-gram vs CBOW

Both architectures learn word vectors, but differently:

- **Skip-gram** (sg=1): predicts surrounding context words from a single center word. Better for infrequent words.
- **CBOW** (sg=0): predicts the center word from averaged context vectors. Faster to train, better for frequent words.

In [ ]:
# Use a subset of text8 for a quicker comparison
subset = sentences_text8[:500]   # 500 sentences for speed

common_params = dict(vector_size=100, window=5, min_count=1, epochs=30, seed=42, workers=4)

model_sg = Word2Vec(sentences=subset, sg=1, **common_params)   # Skip-gram
model_cbow = Word2Vec(sentences=subset, sg=0, **common_params)  # CBOW

test_word = 'science'
if test_word not in model_sg.wv:
    test_word = 'system'

print(f"Skip-gram — most similar to '{test_word}':")
for w, s in model_sg.wv.most_similar(test_word, topn=5):
    print(f"  {w:20s} {s:.4f}")

print(f"\nCBOW — most similar to '{test_word}':")
for w, s in model_cbow.wv.most_similar(test_word, topn=5):
    print(f"  {w:20s} {s:.4f}")

---
## 5. Effect of Hyperparameters

Key Word2Vec hyperparameters and what they control:

| Parameter | Effect |
|---|---|
| `vector_size` | Higher = richer representation, slower training |
| `window` | Larger = broader context, more topical similarity |
| `negative` | More negative samples = better quality, slower |
| `min_count` | Higher = ignore rare words (cleaner vocab) |
| `epochs` | More = better quality, slower |

Let's measure how **window size** affects whether words are syntactically or semantically similar.

In [ ]:
# Compare small vs large window size
# Small window (2): captures syntax (nearby grammar)
# Large window (10): captures semantics (topic-level co-occurrence)

subset = sentences_text8[:2000]

model_w2 = Word2Vec(sentences=subset, vector_size=100, window=2,
                    min_count=2, sg=1, epochs=30, seed=42)
model_w10 = Word2Vec(sentences=subset, vector_size=100, window=10,
                     min_count=2, sg=1, epochs=30, seed=42)

test = 'history'
if test not in model_w2.wv:
    test = 'world'

print(f"Window=2 (syntactic) — similar to '{test}':")
for w, s in model_w2.wv.most_similar(test, topn=5):
    print(f"  {w:20s} {s:.4f}")

print(f"\nWindow=10 (semantic) — similar to '{test}':")
for w, s in model_w10.wv.most_similar(test, topn=5):
    print(f"  {w:20s} {s:.4f}")

---
## 6. Saving and Loading Models

Gensim offers two save formats:

- **Full model** (`save`/`load`): saves weights + training state (can continue training)
- **KeyedVectors only** (`wv.save`/`KeyedVectors.load`): smaller file, inference only

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save the full model (can resume training later)
model.save('models/word2vec_text8.model')
print("Full model saved.")

# Save just the word vectors (smaller, inference only)
model.wv.save('models/word2vec_text8.kv')
print("KeyedVectors saved.")

# --- Load and verify ---
loaded_model = Word2Vec.load('models/word2vec_text8.model')
loaded_kv = KeyedVectors.load('models/word2vec_text8.kv')

# Vectors should be identical
diff = np.max(np.abs(model.wv['king'] - loaded_model.wv['king']))
print(f"\nMax vector diff after reload (should be 0): {diff}")

# Continue training the loaded model on new data
new_sentences = [["the", "duke", "became", "king"], ["the", "duchess", "became", "queen"]]
loaded_model.build_vocab(new_sentences, update=True)
loaded_model.train(new_sentences, total_examples=len(new_sentences), epochs=5)
print("Continued training on 2 new sentences.")
print("'duke' in vocab:", 'duke' in loaded_model.wv)

---
## 7. t-SNE Visualization

Word2Vec vectors live in 100-dimensional space. **t-SNE** projects them to 2D while preserving local structure — words that were close in 100D should stay close in 2D.

In [ ]:
# Pick semantic groups to visualize
word_groups = {
    'Royalty':   ['king', 'queen', 'prince', 'princess', 'throne', 'crown', 'duke', 'emperor'],
    'Countries': ['france', 'germany', 'italy', 'japan', 'china', 'spain', 'india', 'russia'],
    'Animals':   ['dog', 'cat', 'horse', 'lion', 'tiger', 'wolf', 'bear', 'fox'],
    'Science':   ['physics', 'chemistry', 'biology', 'mathematics', 'astronomy', 'geology'],
    'Sports':    ['football', 'tennis', 'basketball', 'swimming', 'baseball', 'cricket'],
}

# Collect valid words (in vocab)
all_words, all_vecs, all_labels = [], [], []
for group, words in word_groups.items():
    for w in words:
        if w in model.wv:
            all_words.append(w)
            all_vecs.append(model.wv[w])
            all_labels.append(group)

vecs = np.array(all_vecs)
print(f"Visualizing {len(all_words)} words across {len(word_groups)} groups")

# t-SNE projection to 2D
tsne = TSNE(n_components=2, perplexity=10, random_state=42, n_iter=1000)
vecs_2d = tsne.fit_transform(vecs)

# Plot
colors = {
    'Royalty': '#e74c3c',
    'Countries': '#3498db',
    'Animals': '#2ecc71',
    'Science': '#9b59b6',
    'Sports': '#f39c12',
}

fig, ax = plt.subplots(figsize=(13, 9))
for group in word_groups:
    idx = [i for i, l in enumerate(all_labels) if l == group]
    ax.scatter(vecs_2d[idx, 0], vecs_2d[idx, 1],
               c=colors[group], label=group, s=120, alpha=0.8)
    for i in idx:
        ax.annotate(all_words[i],
                    (vecs_2d[i, 0] + 0.5, vecs_2d[i, 1] + 0.5),
                    fontsize=9)

ax.legend(fontsize=12)
ax.set_title("t-SNE of Word2Vec Embeddings (trained on text8)", fontsize=14)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()
print("Words from the same semantic category cluster together!")

---
## 8. Visualizing the Training Process

Let's watch how word vectors improve as training progresses by training multiple models with increasing epochs and tracking the analogy quality.

In [ ]:
# Track how similarity of known pairs changes with training epochs
subset = sentences_text8[:3000]
epoch_list = [1, 2, 5, 10, 20, 30]

# Pairs that should be similar (positive) and dissimilar (negative)
similar_pairs = [('king', 'queen'), ('france', 'germany'), ('dog', 'cat')]
dissimilar_pairs = [('king', 'pizza'), ('france', 'dog'), ('tennis', 'astronomy')]

results = {pair: [] for pair in similar_pairs + dissimilar_pairs}

for epochs in epoch_list:
    m = Word2Vec(sentences=subset, vector_size=100, window=5,
                 min_count=2, sg=1, epochs=epochs, seed=42)
    for pair in similar_pairs + dissimilar_pairs:
        w1, w2 = pair
        if w1 in m.wv and w2 in m.wv:
            results[pair].append(m.wv.similarity(w1, w2))
        else:
            results[pair].append(None)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for pair in similar_pairs:
    vals = results[pair]
    if any(v is not None for v in vals):
        ax1.plot(epoch_list, vals, marker='o', label=f"{pair[0]}-{pair[1]}")
ax1.set_title("Similar pairs — similarity should increase")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Cosine similarity")
ax1.legend()

for pair in dissimilar_pairs:
    vals = results[pair]
    if any(v is not None for v in vals):
        ax2.plot(epoch_list, vals, marker='o', label=f"{pair[0]}-{pair[1]}")
ax2.set_title("Dissimilar pairs — similarity should stay low")
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Cosine similarity")
ax2.legend()

plt.suptitle("Word Similarity vs Training Epochs", fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Gender Bias in Word Vectors

Word vectors trained on real text encode **societal biases** present in the corpus.  
A classic example: profession words projected onto a "gender direction".

In [ ]:
# Gender direction: he - she (points from female-associated to male-associated)
if 'he' in model.wv and 'she' in model.wv:
    gender_dir = model.wv['he'] - model.wv['she']
    gender_dir = gender_dir / np.linalg.norm(gender_dir)

    professions = [
        'doctor', 'nurse', 'engineer', 'teacher', 'scientist',
        'secretary', 'programmer', 'librarian', 'pilot', 'receptionist',
        'professor', 'dancer', 'architect', 'soldier',
    ]

    bias_scores = []
    for prof in professions:
        if prof in model.wv:
            score = float(np.dot(model.wv[prof], gender_dir))
            bias_scores.append((prof, score))

    bias_scores.sort(key=lambda x: x[1])

    fig, ax = plt.subplots(figsize=(10, 6))
    names  = [b[0] for b in bias_scores]
    scores = [b[1] for b in bias_scores]
    bar_colors = ['#e74c3c' if s > 0 else '#3498db' for s in scores]

    ax.barh(names, scores, color=bar_colors)
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_xlabel('← female-associated   |   male-associated →')
    ax.set_title('Gender Bias in Word2Vec Profession Vectors')
    plt.tight_layout()
    plt.show()

    print("\nKey insight: These biases reflect patterns in the Wikipedia training text,")
    print("not objective truth. Being aware of them is critical in production NLP systems.")
else:
    print("'he'/'she' not in vocabulary (corpus too small).")

---
## Summary

### What we covered

| Topic | Key takeaway |
|---|---|
| **Training** | `Word2Vec(sentences, vector_size, window, sg, epochs)` |
| **Skip-gram vs CBOW** | Skip-gram better for rare words; CBOW faster |
| **most_similar** | Cosine nearest neighbors in vector space |
| **Vector arithmetic** | king − man + woman ≈ queen |
| **doesnt_match** | Odd-one-out detection |
| **Save/load** | `model.save()` for full model, `model.wv.save()` for inference only |
| **Hyperparameters** | Window size controls syntactic vs semantic similarity |
| **t-SNE** | Semantic clusters visible in 2D projection |
| **Bias** | Training corpora encode societal biases into vectors |

### References

- Mikolov et al. (2013) — [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781) (Word2Vec original paper)
- Mikolov et al. (2013) — [Distributed Representations of Words and Phrases](https://arxiv.org/abs/1310.4546) (negative sampling)
- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)

### Next steps

- **GloVe**: global co-occurrence matrix factorization (see `word_vec_visualization.ipynb`)
- **FastText**: subword embeddings — handles unseen/misspelled words
- **Sentence-BERT / OpenAI embeddings**: modern sentence-level representations